In [1]:
import pandas as pd
import numpy as np
from scipy.stats import linregress
from xgboost import XGBClassifier, XGBRegressor
from sklearn.model_selection import StratifiedKFold, KFold, cross_val_score, GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (
    accuracy_score, average_precision_score,
    mean_squared_error, classification_report, confusion_matrix
)
from imblearn.over_sampling import SMOTE
import pickle
import joblib
import os
import time
import warnings
warnings.filterwarnings('ignore')

# ══════════════════════════════════════════════════════════════════════════════
# STEP 1 — Load data and aggregate per subject
# 2,553 trial rows → 529 subjects × 12 features
# ══════════════════════════════════════════════════════════════════════════════
df = pd.read_csv('AI_MSHM_HRV_rPPG.csv')

def safe_std(x):
    vals = x.dropna().tolist()
    return np.std(vals, ddof=1) if len(vals) >= 2 else np.nan

def safe_slope(x):
    vals = x.dropna().tolist()
    if len(vals) < 2: return np.nan
    from scipy.stats import linregress
    return round(linregress(range(len(vals)), vals).slope, 4)

# ── Per-subject longitudinal aggregation ──────────────────────────────────────
# Each subject has up to 5 trial sessions (morning, evening, trial_001/002/003)
# We aggregate across all sessions to get a stable subject-level signal
subject = df.groupby('Subject_ID').agg(
    RMSSD_mean   = ('RMSSD',                'mean'),
    RMSSD_min    = ('RMSSD',                'min'),
    RMSSD_max    = ('RMSSD',                'max'),
    RMSSD_std    = ('RMSSD',                safe_std),
    Temp_mean    = ('Mean_Temp',             'mean'),
    Temp_std     = ('Mean_Temp',             safe_std),
    EDA_mean     = ('Mean_EDA',              'mean'),
    EDA_max      = ('Mean_EDA',              'max'),
    EDA_std      = ('Mean_EDA',              safe_std),
    ASI_mean     = ('Autonomic_Stress_Index', 'mean'),
    ASI_max      = ('Autonomic_Stress_Index', 'max'),
    n_trials     = ('Trial',                 'count'),
    label        = ('Label',                 'max'),
    # label = max across trials: if ANY trial is label=1, subject is label=1
).reset_index()

print(f'Subjects: {len(subject)}  |  Label distribution: {subject["label"].value_counts().to_dict()}')
print(f'Imbalance ratio: {subject["label"].value_counts()[0]/subject["label"].value_counts()[1]:.1f}:1')
print(f'Features per subject: {subject.shape[1] - 2}')  # exclude Subject_ID and label

# ══════════════════════════════════════════════════════════════════════════════
# STEP 2 — Intermediate flags (clinical threshold rules)
# ══════════════════════════════════════════════════════════════════════════════
subject['LowRMSSD_Flag']    = (subject['RMSSD_mean'] < 20).astype(int)
    # RMSSD < 20ms = chronic SNS activation = insulin resistance biomarker

subject['ModLowRMSSD_Flag'] = (subject['RMSSD_mean'] < 30).astype(int)
    # RMSSD < 30ms = moderate HRV suppression = elevated cardiovascular risk

subject['HighEDA_Flag']     = (subject['EDA_mean'] > 5.0).astype(int)
    # Mean EDA > 5.0 µS = persistent sympathetic overload = chronic stress

subject['ElevatedEDA_Flag'] = (subject['EDA_mean'] > 2.0).astype(int)
    # Mean EDA > 2.0 µS = elevated baseline sympathetic activation

subject['HighTemp_Flag']    = (subject['Temp_mean'] > 37.0).astype(int)
    # Skin temp > 37°C = febrile / metabolic stress pattern

subject['LowTemp_Flag']     = (subject['Temp_mean'] < 30.0).astype(int)
    # Skin temp < 30°C = reduced peripheral perfusion = organ stress signal

subject['HighASI_Flag']     = (subject['ASI_mean'] > 0.10).astype(int)
    # ASI > 0.10 = above cohort 90th percentile = high autonomic burden

print('\nFlag prevalence:')
for flag in ['LowRMSSD_Flag','ModLowRMSSD_Flag','HighEDA_Flag','ElevatedEDA_Flag',
             'HighTemp_Flag','LowTemp_Flag','HighASI_Flag']:
    n   = int(subject[flag].sum())
    pct = subject[flag].mean() * 100
    print(f'  {flag:<24}  {n:3} subjects  ({pct:.1f}%)')

# ══════════════════════════════════════════════════════════════════════════════
# STEP 3 — Compute all 6 disease SCORES, FLAGS, and SEVERITY CATEGORIES
#
# Severity scale (uniform across all diseases):
#   0.00 – 0.19  →  Minimal   (no clinical concern)
#   0.20 – 0.39  →  Mild      (monitor — lifestyle guidance)
#   0.40 – 0.59  →  Moderate  (alert — medical review recommended)
#   0.60 – 0.79  →  Severe    (referral — specialist review required)
#   0.80 – 1.00  →  Extreme   (urgent — immediate clinical intervention)
# ══════════════════════════════════════════════════════════════════════════════

SEVERITY_BINS   = [-0.001, 0.19, 0.39, 0.59, 0.79, 1.001]
SEVERITY_LABELS = ['Minimal', 'Mild', 'Moderate', 'Severe', 'Extreme']

def severity(series):
    return pd.cut(series, bins=SEVERITY_BINS, labels=SEVERITY_LABELS)

# ── 1. Chronic Stress ─────────────────────────────────────────────────────────
# EDA is the direct sympathetic stress marker (primary)
# RMSSD inversion: low HRV = parasympathetic cannot suppress stress
# Temp deviation: skin temp reflects sympatho-adrenal activation
subject['Stress_Score'] = (
    (subject['EDA_mean'] / 16).clip(0, 1)              * 0.45 +
    (1 - subject['RMSSD_mean'] / 100).clip(0, 1)       * 0.35 +
    (subject['Temp_mean'] - 36.5).abs() / 3.5          * 0.20
).round(4)
subject['Stress_Flag']     = (subject['Stress_Score'] >= 0.35).astype(int)
subject['Stress_Severity'] = severity(subject['Stress_Score'])

# ── 2. Cardiovascular Disease ─────────────────────────────────────────────────
# RMSSD is the dominant CVD signal — 30-day declining trend predicts cardiac events
# Low RMSSD = chronic SNS = endothelial damage = atherosclerosis
subject['CVD_Score'] = (
    (1 - subject['RMSSD_mean'] / 100).clip(0, 1)       * 0.55 +
    (subject['EDA_mean'] / 16).clip(0, 1)               * 0.25 +
    (subject['Temp_mean'] - 36.5).abs() / 3.5           * 0.20
).round(4)
subject['CVD_Flag']     = (subject['CVD_Score'] >= 0.40).astype(int)
subject['CVD_Severity'] = severity(subject['CVD_Score'])

# ── 3. Type 2 Diabetes — ANS Component ───────────────────────────────────────
# RMSSD inversion → insulin resistance pathway (Tekin et al. 2010)
# High EDA → cortisol elevation → insulin receptor suppression
subject['T2D_Score'] = (
    (1 - subject['RMSSD_mean'] / 100).clip(0, 1)       * 0.60 +
    (subject['EDA_mean'] / 16).clip(0, 1)               * 0.40
).round(4)
subject['T2D_Flag']     = (subject['T2D_Score'] >= 0.40).astype(int)
subject['T2D_Severity'] = severity(subject['T2D_Score'])

# ── 4. Metabolic Syndrome ────────────────────────────────────────────────────
# NCEP Criterion 5 proxy: ANS dysregulation = insulin resistance = central adiposity
subject['Metabolic_Score'] = (
    (1 - subject['RMSSD_mean'] / 100).clip(0, 1)       * 0.50 +
    (subject['EDA_mean'] / 16).clip(0, 1)               * 0.30 +
    (subject['Temp_mean'] - 36.5).abs() / 3.5           * 0.20
).round(4)
subject['Metabolic_Flag']     = (subject['Metabolic_Score'] >= 0.40).astype(int)
subject['Metabolic_Severity'] = severity(subject['Metabolic_Score'])

# ── 5. Heart Failure / Organ Damage ──────────────────────────────────────────
# Very low RMSSD (<15ms sustained) = near-total vagal withdrawal = cardiac decompensation
# Uses tighter RMSSD ceiling (50ms not 100ms) to capture extreme suppression
# Low peripheral temp = reduced perfusion = organ stress
subject['HeartFailure_Score'] = (
    (1 - subject['RMSSD_mean'] / 50).clip(0, 1)        * 0.55 +
    (subject['EDA_mean'] / 16).clip(0, 1)               * 0.25 +
    ((30 - subject['Temp_mean'].clip(28, 30)) / 2)      * 0.20
).round(4)
subject['HeartFailure_Flag']     = (subject['HeartFailure_Score'] >= 0.35).astype(int)
subject['HeartFailure_Severity'] = severity(subject['HeartFailure_Score'])

# ── 6. Infertility — ANS Component ───────────────────────────────────────────
# RMSSD suppression → insulin resistance → androgen excess → anovulation
# EDA elevation → cortisol → LH pulse disruption → follicle selection failure
subject['Infertility_Score'] = (
    (1 - subject['RMSSD_mean'] / 100).clip(0, 1)       * 0.60 +
    (subject['EDA_mean'] / 16).clip(0, 1)               * 0.40
).round(4)
subject['Infertility_Flag']     = (subject['Infertility_Score'] >= 0.40).astype(int)
subject['Infertility_Severity'] = severity(subject['Infertility_Score'])

# ── Severity distribution summary ─────────────────────────────────────────────
print('=' * 72)
print('  Severity scale:  0.00–0.19 Minimal | 0.20–0.39 Mild | 0.40–0.59 Moderate')
print('                   0.60–0.79 Severe  | 0.80–1.00 Extreme')
print('=' * 72)
print(f'  {"Disease":<18} {"Minimal":>9} {"Mild":>9} {"Moderate":>10} {"Severe":>9} {"Extreme":>9}')
print(f'  {"-"*68}')
sev_map = {
    'Stress'       : 'Stress_Severity',
    'CVD'          : 'CVD_Severity',
    'T2D'          : 'T2D_Severity',
    'Metabolic'    : 'Metabolic_Severity',
    'HeartFailure' : 'HeartFailure_Severity',
    'Infertility'  : 'Infertility_Severity',
}
for disease, sev_col in sev_map.items():
    counts = subject[sev_col].value_counts().reindex(SEVERITY_LABELS, fill_value=0)
    n = len(subject)
    print(
        f'  {disease:<18} '
        f'{counts["Minimal"]:>4}({counts["Minimal"]/n*100:.0f}%)  '
        f'{counts["Mild"]:>4}({counts["Mild"]/n*100:.0f}%)  '
        f'{counts["Moderate"]:>5}({counts["Moderate"]/n*100:.0f}%)  '
        f'{counts["Severe"]:>4}({counts["Severe"]/n*100:.0f}%)  '
        f'{counts["Extreme"]:>4}({counts["Extreme"]/n*100:.0f}%)'
    )

# ══════════════════════════════════════════════════════════════════════════════
# STEP 4 — Feature matrix
# ══════════════════════════════════════════════════════════════════════════════
# Drop ASI from X — it is EDA-derived (r=0.786) and would leak EDA into models
# ASI is used only as the Y target in the regressor
FEATURES = [
    'RMSSD_mean', 'RMSSD_min', 'RMSSD_std',
    'Temp_mean',  'Temp_std',
    'EDA_mean',   'EDA_max',  'EDA_std',
    'n_trials',
]

X        = subject[FEATURES]
X_filled = X.fillna(X.median())

DISEASES = {
    'Stress'      : ('Stress_Flag',       'Stress_Score',       'Stress_Severity'),
    'CVD'         : ('CVD_Flag',          'CVD_Score',          'CVD_Severity'),
    'T2D'         : ('T2D_Flag',          'T2D_Score',          'T2D_Severity'),
    'Metabolic'   : ('Metabolic_Flag',    'Metabolic_Score',    'Metabolic_Severity'),
    'HeartFailure': ('HeartFailure_Flag', 'HeartFailure_Score', 'HeartFailure_Severity'),
    'Infertility' : ('Infertility_Flag',  'Infertility_Score',  'Infertility_Severity'),
}

print(f'\nFeature matrix: {X_filled.shape[0]} subjects × {X_filled.shape[1]} features')
print(f'Missing values: {X_filled.isna().sum().sum()}')

# ══════════════════════════════════════════════════════════════════════════════
# STEP 5 — Isolation Forest (anomaly detection — no label required)
#
# Handles the 10:1 trial-level imbalance natively.
# contamination = proportion of trials expected to be anomalous (~9%)
# ══════════════════════════════════════════════════════════════════════════════
from sklearn.ensemble import IsolationForest

scaler_iso = StandardScaler()
X_scaled   = scaler_iso.fit_transform(X_filled)

iso = IsolationForest(
    n_estimators  = 200,
    contamination = 0.09,   # ~9% of subjects carry label=1 pattern
    max_features  = 1.0,
    random_state  = 42,
)
iso.fit(X_scaled)

subject['Anomaly_Score'] = iso.decision_function(X_scaled)   # lower = more anomalous
subject['Anomaly_Flag']  = (iso.predict(X_scaled) == -1).astype(int)

print(f'\nIsolation Forest — anomalies detected: {subject["Anomaly_Flag"].sum()} subjects ')
print(f'({subject["Anomaly_Flag"].mean()*100:.1f}% flagged as anomalous)')

# ══════════════════════════════════════════════════════════════════════════════
# STEP 6 — Train XGBoost Classifier + Regressor per disease
#
# Classifier: predicts binary flag (is this subject at risk for disease X?)
# Regressor:  predicts continuous risk score (how severe is the risk 0–1?)
# GroupKFold by Subject_ID prevents trial-level leakage across folds
# ══════════════════════════════════════════════════════════════════════════════
results = {}

print('\n' + '=' * 70)
print('AI-MSHM  |  rPPG / HRV — Full ML Pipeline Results')
print('=' * 70)

for disease, (flag_col, score_col, sev_col) in DISEASES.items():

    y_flag  = subject[flag_col]
    y_score = subject[score_col]

    pos   = int(y_flag.sum())
    neg   = len(y_flag) - pos
    ratio = neg / pos if pos > 0 else 1

    # ── A: Classification ────────────────────────────────────────────────────
    if pos >= 6:
        sm = SMOTE(k_neighbors=min(5, pos - 1), random_state=42)
        X_res, y_res = sm.fit_resample(X_filled, y_flag)
    else:
        X_res, y_res = X_filled, y_flag

    clf = XGBClassifier(
        scale_pos_weight = ratio,
        n_estimators     = 300,
        max_depth        = 3,
        learning_rate    = 0.05,
        subsample        = 0.8,
        colsample_bytree = 1.0,
        min_child_weight = 3,
        reg_alpha        = 0.1,
        eval_metric      = 'aucpr',
        random_state     = 42,
        verbosity        = 0,
    )

    cv_clf        = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    pr_auc_scores = cross_val_score(clf, X_res, y_res, cv=cv_clf, scoring='average_precision')
    acc_scores    = cross_val_score(clf, X_res, y_res, cv=cv_clf, scoring='accuracy')

    clf.fit(X_res, y_res)
    y_pred_flag  = clf.predict(X_filled)
    y_pred_proba = clf.predict_proba(X_filled)[:, 1]
    acc_full     = accuracy_score(y_flag, y_pred_flag)

    # ── B: Regression ────────────────────────────────────────────────────────
    reg = XGBRegressor(
        n_estimators     = 300,
        max_depth        = 3,
        learning_rate    = 0.05,
        subsample        = 0.8,
        colsample_bytree = 1.0,
        reg_alpha        = 0.1,
        random_state     = 42,
        verbosity        = 0,
    )

    cv_reg         = KFold(n_splits=5, shuffle=True, random_state=42)
    neg_mse_scores = cross_val_score(reg, X_filled, y_score, cv=cv_reg, scoring='neg_mean_squared_error')
    rmse_cv        = float(np.sqrt(-neg_mse_scores.mean()))

    reg.fit(X_filled, y_score)
    y_pred_score = reg.predict(X_filled)
    rmse_full    = float(np.sqrt(mean_squared_error(y_score, y_pred_score)))

    results[disease] = {
        'clf':            clf,
        'reg':            reg,
        'pr_auc':         round(float(pr_auc_scores.mean()), 4),
        'accuracy_cv':    round(float(acc_scores.mean()), 4),
        'accuracy_full':  round(acc_full, 4),
        'rmse_cv':        round(rmse_cv, 4),
        'rmse_full':      round(rmse_full, 4),
        'y_pred_flag':    y_pred_flag,
        'y_pred_proba':   y_pred_proba,
        'y_pred_score':   y_pred_score,
        'positives':      pos,
    }

    print(f'\n{"─"*70}')
    print(f'  {disease}  (positives = {pos} / {len(y_flag)})')
    print(f'{"─"*70}')
    print(f'  Classification (XGBoostClassifier)')
    print(f'    PR-AUC    (CV)  : {pr_auc_scores.mean():.4f}  ± {pr_auc_scores.std():.4f}')
    print(f'    Accuracy  (CV)  : {acc_scores.mean()*100:.2f}%  ± {acc_scores.std()*100:.2f}%')
    print(f'    Accuracy  (full): {acc_full*100:.2f}%')
    print(f'  Regression (XGBoostRegressor — risk score 0–1)')
    print(f'    RMSE      (CV)  : {rmse_cv:.4f}')
    print(f'    RMSE      (full): {rmse_full:.4f}')
    print(f'\n  Confusion matrix (on full dataset):')
    cm = confusion_matrix(y_flag, y_pred_flag)
    print(f'    TN={cm[0,0]}  FP={cm[0,1]}  FN={cm[1,0]}  TP={cm[1,1]}')

# ══════════════════════════════════════════════════════════════════════════════
# STEP 7 — Per-subject prediction table (with severity categories)
# ══════════════════════════════════════════════════════════════════════════════
print(f'\n{"="*72}')
print('  Per-subject risk score predictions — first 20 subjects shown')
print(f'{"="*72}')

output = subject[['Subject_ID', 'label']].copy()
output.index = range(len(output))

for disease, (flag_col, score_col, sev_col) in DISEASES.items():
    r = results[disease]
    output[f'{disease}_ActualFlag']     = subject[flag_col].values
    output[f'{disease}_ActualScore']    = subject[score_col].values
    output[f'{disease}_ActualSeverity'] = subject[sev_col].values
    output[f'{disease}_PredFlag']       = r['y_pred_flag']
    output[f'{disease}_PredScore']      = r['y_pred_score'].round(4)
    output[f'{disease}_RiskProb']       = r['y_pred_proba'].round(4)
    output[f'{disease}_PredSeverity']   = pd.cut(
        output[f'{disease}_PredScore'],
        bins=SEVERITY_BINS, labels=SEVERITY_LABELS
    )

for disease in DISEASES:
    cols = ['Subject_ID',
            f'{disease}_ActualFlag', f'{disease}_ActualScore', f'{disease}_ActualSeverity',
            f'{disease}_PredFlag',   f'{disease}_PredScore',   f'{disease}_PredSeverity',
            f'{disease}_RiskProb']
    print(f'\n  {disease}:')
    print(f'  {"Subject":<10} {"ActFlg":<8} {"ActScr":<9} {"ActSev":<12} {"PredFlg":<9} {"PredScr":<9} {"PredSev":<12} {"RiskProb"}')
    print(f'  {"-"*80}')
    for _, row in output[cols].head(20).iterrows():
        print(
            f'  {int(row["Subject_ID"]):<10} '
            f'{int(row[f"{disease}_ActualFlag"]):<8} '
            f'{row[f"{disease}_ActualScore"]:<9.4f} '
            f'{str(row[f"{disease}_ActualSeverity"]):<12} '
            f'{int(row[f"{disease}_PredFlag"]):<9} '
            f'{row[f"{disease}_PredScore"]:<9.4f} '
            f'{str(row[f"{disease}_PredSeverity"]):<12} '
            f'{row[f"{disease}_RiskProb"]:.4f}'
        )

# ══════════════════════════════════════════════════════════════════════════════
# STEP 8 — Save output CSVs (Windows-safe)
# ══════════════════════════════════════════════════════════════════════════════
def safe_save(df, filename, max_retries=3):
    for attempt in range(1, max_retries + 1):
        try:
            df.to_csv(filename, index=False)
            print(f'  Saved: {filename}')
            return filename
        except PermissionError:
            if attempt < max_retries:
                print(f'  PermissionError — file open elsewhere. Retrying in 3s... ({attempt}/{max_retries})')
                time.sleep(3)
            else:
                ts       = time.strftime('%Y%m%d_%H%M%S')
                fallback = filename.replace('.csv', f'_{ts}.csv')
                df.to_csv(fallback, index=False)
                print(f'  Saved fallback: {fallback}')
                return fallback

safe_save(output, 'rppg_hrv_predictions_full.csv')

metrics_rows = []
for disease, (flag_col, score_col, sev_col) in DISEASES.items():
    r = results[disease]
    metrics_rows.append({
        'Disease'      : disease,
        'Positives'    : r['positives'],
        'PR_AUC_CV'    : r['pr_auc'],
        'Accuracy_CV'  : f'{r["accuracy_cv"]*100:.2f}%',
        'Accuracy_Full': f'{r["accuracy_full"]*100:.2f}%',
        'RMSE_CV'      : r['rmse_cv'],
        'RMSE_Full'    : r['rmse_full'],
    })
safe_save(pd.DataFrame(metrics_rows), 'rppg_hrv_metrics_summary.csv')

# ══════════════════════════════════════════════════════════════════════════════
# STEP 9 — Single-bundle Pickle + Joblib serialisation
#
# ONE file per format — each contains the complete pipeline:
#
#   models/ai_mshm_rppg_pipeline.pkl      (Pickle  — portable)
#   models/ai_mshm_rppg_pipeline.joblib   (Joblib  — recommended for deployment)
#
# Bundle keys:
#   classifiers         : { disease: XGBoostClassifier }   6 classifiers
#   regressors          : { disease: XGBoostRegressor  }   6 regressors
#   isolation_forest    : IsolationForest (anomaly detection model)
#   scaler              : StandardScaler fitted on X_filled
#   scaler_iso          : StandardScaler fitted for IsolationForest
#   feature_names       : ordered list of 9 input features
#   flag_thresholds     : { disease: float score cutoff }
#   severity_bins       : [-0.001, 0.19, 0.39, 0.59, 0.79, 1.001]
#   severity_labels     : [Minimal, Mild, Moderate, Severe, Extreme]
#   diseases            : ordered list of 6 disease names
#   model_metrics       : { disease: { pr_auc, accuracy_cv, rmse_cv, ... } }
#   trained_at          : ISO timestamp
#   dataset / module / n_subjects / n_diseases
# ══════════════════════════════════════════════════════════════════════════════

os.makedirs('models', exist_ok=True)
PICKLE_PATH = os.path.join('models', 'ai_mshm_rppg_pipeline.pkl')
JOBLIB_PATH = os.path.join('models', 'ai_mshm_rppg_pipeline.joblib')

# Fit scaler on training feature matrix
scaler = StandardScaler()
scaler.fit(X_filled)

# Build single pipeline bundle
pipeline_bundle = {

    # ── Models ────────────────────────────────────────────────────────────────
    'classifiers'      : {disease: results[disease]['clf'] for disease in DISEASES},
    'regressors'       : {disease: results[disease]['reg'] for disease in DISEASES},
    'isolation_forest' : iso,           # anomaly detector — no label required

    # ── Preprocessing ─────────────────────────────────────────────────────────
    'scaler'           : scaler,         # for XGBoost models
    'scaler_iso'       : scaler_iso,     # for Isolation Forest
    'feature_names'    : FEATURES,

    # ── Thresholds and categories ─────────────────────────────────────────────
    'flag_thresholds': {
        'Stress'       : 0.35,
        'CVD'          : 0.40,
        'T2D'          : 0.40,
        'Metabolic'    : 0.40,
        'HeartFailure' : 0.35,
        'Infertility'  : 0.40,
    },
    'severity_bins'  : SEVERITY_BINS,
    'severity_labels': SEVERITY_LABELS,

    # ── Registry ──────────────────────────────────────────────────────────────
    'diseases'      : list(DISEASES.keys()),

    # ── Performance metrics per disease ───────────────────────────────────────
    'model_metrics': {
        disease: {
            'pr_auc'       : results[disease]['pr_auc'],
            'accuracy_cv'  : results[disease]['accuracy_cv'],
            'accuracy_full': results[disease]['accuracy_full'],
            'rmse_cv'      : results[disease]['rmse_cv'],
            'rmse_full'    : results[disease]['rmse_full'],
            'positives'    : results[disease]['positives'],
            'total'        : len(subject),
        }
        for disease in DISEASES
    },

    # ── Provenance ────────────────────────────────────────────────────────────
    'trained_at' : time.strftime('%Y-%m-%dT%H:%M:%S'),
    'dataset'    : 'AI_MSHM_HRV_rPPG.csv',
    'module'     : 'rPPG — Passive Layer 1 (HRV + EDA + Temperature)',
    'n_subjects' : len(subject),
    'n_diseases' : len(DISEASES),
}

# ── A: Pickle ─────────────────────────────────────────────────────────────────
with open(PICKLE_PATH, 'wb') as f:
    pickle.dump(pipeline_bundle, f, protocol=pickle.HIGHEST_PROTOCOL)
pkl_kb = os.path.getsize(PICKLE_PATH) / 1024
print(f'\nPickle  saved → {PICKLE_PATH}  ({pkl_kb:.1f} KB)')

# ── B: Joblib (compress=3 — balanced speed/size) ──────────────────────────────
joblib.dump(pipeline_bundle, JOBLIB_PATH, compress=3)
jbl_kb = os.path.getsize(JOBLIB_PATH) / 1024
print(f'Joblib  saved → {JOBLIB_PATH}  ({jbl_kb:.1f} KB)')

# ── C: Verification — reload both bundles, predict on 5 sample subjects ───────
print('\n' + '=' * 65)
print('  Verification — reload both bundles and predict on 5 subjects')
print('=' * 65)

X_sample = X_filled.iloc[:5]

for fmt, path in [('Pickle', PICKLE_PATH), ('Joblib', JOBLIB_PATH)]:
    if fmt == 'Pickle':
        with open(path, 'rb') as f:
            loaded = pickle.load(f)
    else:
        loaded = joblib.load(path)

    expected = {'classifiers','regressors','isolation_forest','scaler','scaler_iso',
                'feature_names','flag_thresholds','severity_bins','severity_labels','diseases'}
    missing  = expected - set(loaded.keys())
    print(f'\n  {fmt} reload — keys OK: {len(missing)==0}  |  ')
    print(f'  Classifiers: {len(loaded["classifiers"])}  |  IsolationForest: {type(loaded["isolation_forest"]).__name__}')
    print(f'  {"Disease":<16} {"RiskProb[0]":>13} {"PredScore[0]":>14} {"Severity":<12} {"Flag"}')
    print(f'  {"-"*62}')
    for disease in loaded['diseases']:
        prob  = round(float(loaded['classifiers'][disease].predict_proba(X_sample)[:,1][0]), 4)
        score = round(float(loaded['regressors'][disease].predict(X_sample)[0]), 4)
        sev   = pd.cut([score], bins=loaded['severity_bins'],
                        labels=loaded['severity_labels'])[0]
        flag  = int(score >= loaded['flag_thresholds'][disease])
        print(f'  {disease:<16} {prob:>13.4f} {score:>14.4f} {str(sev):<12} flag={flag}')

    # Isolation Forest check
    X_sample_scaled = loaded['scaler_iso'].transform(X_sample)
    anomaly_flags   = (loaded['isolation_forest'].predict(X_sample_scaled) == -1).astype(int)
    print(f'  IsolationForest anomaly flags on 5 samples: {list(anomaly_flags)}')

# ── D: Bundle contents summary ────────────────────────────────────────────────
print('\n' + '=' * 65)
print('  Bundle contents')
print('=' * 65)
print(f'  {"Key":<22} {"Type":<30} {"Details"}')
print(f'  {"-"*70}')
for k, v in pipeline_bundle.items():
    vtype = type(v).__name__
    if k == 'classifiers':
        detail = f'{len(v)} diseases: {list(v.keys())[:3]}...'
    elif k == 'regressors':
        detail = f'{len(v)} diseases'
    elif k == 'isolation_forest':
        detail = f'contamination={v.contamination}  n_estimators={v.n_estimators}'
    elif k == 'feature_names':
        detail = f'{len(v)} features: {v[:3]}...'
    elif k == 'model_metrics':
        detail = 'pr_auc, accuracy_cv, rmse_cv, positives per disease'
    elif k == 'severity_bins':
        detail = str(v)
    elif k == 'severity_labels':
        detail = str(v)
    elif k == 'flag_thresholds':
        detail = str(v)
    elif isinstance(v, dict):
        detail = f'{len(v)} keys'
    else:
        detail = str(v)[:55]
    print(f'  {k:<22} {vtype:<30} {detail}')

print(f'\n  Pickle  file size : {pkl_kb:.1f} KB  →  {PICKLE_PATH}')
print(f'  Joblib  file size : {jbl_kb:.1f} KB  →  {JOBLIB_PATH}')
print('\nDone. Pipeline bundles ready for deployment.')


Subjects: 529  |  Label distribution: {0: 300, 1: 229}
Imbalance ratio: 1.3:1
Features per subject: 12

Flag prevalence:
  LowRMSSD_Flag               1 subjects  (0.2%)
  ModLowRMSSD_Flag            4 subjects  (0.8%)
  HighEDA_Flag               17 subjects  (3.2%)
  ElevatedEDA_Flag          184 subjects  (34.8%)
  HighTemp_Flag               3 subjects  (0.6%)
  LowTemp_Flag                1 subjects  (0.2%)
  HighASI_Flag               38 subjects  (7.2%)
  Severity scale:  0.00–0.19 Minimal | 0.20–0.39 Mild | 0.40–0.59 Moderate
                   0.60–0.79 Severe  | 0.80–1.00 Extreme
  Disease              Minimal      Mild   Moderate    Severe   Extreme
  --------------------------------------------------------------------
  Stress               27(5%)   400(76%)    102(19%)     0(0%)     0(0%)
  CVD                  20(4%)   247(47%)    258(49%)     4(1%)     0(0%)
  T2D                  99(19%)   340(64%)     88(17%)     2(0%)     0(0%)
  Metabolic            20(4%)   281(53%)